Required Packages

In [3]:
import numpy as np
import pulp
import pandas as pd

## Incredible Chairs 2

What if the company were not producing two different chairs on three production lines, but say ten different chairs on five different production lines? If the same approach were used, the size of the LP model would increase, and it would be quite tedious to input using the direct approach (1.1). In this case, we can formulate a more abstract model.

Given a set of chairs $C$, indexed by $c$, and a set of production lines $P$, indexed by $p$, we can define:

- a (one-dimensional) vector $Profit_c$, which states the profit for each chair $c \in C$,
- a (one-dimensional) production capacity vector $Capacity_p$, which states the production capacity of each production line $p \in P$,
- a (two-dimensional) matrix $ResourceUsage_{p,c}$ that states how much of the available resource on production line $p$ is needed for production of one unit of chair $c$.

This gives the following LP model:

$$
\begin{aligned}
\text{Maximize} \quad & \sum_{c \in C} Profit_c \cdot x_c \\
\text{subject to} \quad & \sum_{c \in C} ResourceUsage_{p,c} \cdot x_c \le Capacity_p, && \forall p \in P, \\
& x_c \ge 0, && \forall c \in C.
\end{aligned}
$$

In [4]:
chairs = ['A', 'B']
lines = ['Line 1', 'Line 2', 'Line 3']

profit = np.array([4, 6]) #Profit c
capacity = np.array([14, 15, 36]) #Production capacity p

usage = np.array([[2, 0], [0, 3], [4, 3]]) #Usage matrix a


In [9]:
def solve_chairs(profit, capacity, usage, chairs, lines):
    profit = np.asarray(profit, dtype=float)
    capacity = np.asarray(capacity, dtype=float)
    usage = np.asarray(usage, dtype=float)


    # 1. Empty model with a name and a direction
    model = pulp.LpProblem("Incredible_Chairs_2", pulp.LpMaximize)

    # 2. Decision variables: one x_c per chair, x_c >= 0
    x = model.add_variable_dict("x", chairs, lowBound=0)

    # 3. Objective: sum over c of Profit_c * x_c
    model += pulp.lpSum(profit[j] * x[c] for j, c in enumerate(chairs))

    # 4. One capacity constraint per production line p
    for i, p in enumerate(lines):
        model += (
            pulp.lpSum(usage[i, j] * x[c] for j, c in enumerate(chairs)) <= capacity[i],
            f"Capacity_{p}",
        )

    # 5. Solve
    result = model.solve(pulp.HiGHS(msg=False))
    return model, x, result

In [10]:
model, x, result = solve_chairs(profit, capacity, usage, chairs, lines)

print(model)                 # the LP written out, to check it matches the formulation
print(result.status_str)     # "Optimal"
print(result.objective)      # 51.0

Incredible_Chairs_2:
MAXIMIZE
4.0*x_A + 6.0*x_B + 0.0
SUBJECT TO
Capacity_Line_1: 2 x_A <= 14

Capacity_Line_2: 3 x_B <= 15

Capacity_Line_3: 4 x_A + 3 x_B <= 36

VARIABLES
x_A Continuous
x_B Continuous

Optimal
51.0


In [11]:
x_val = np.array([x[c].value() for c in chairs])

production = pd.DataFrame({"chair": chairs, "quantity": x_val, "profit": profit * x_val})
lines_used = pd.DataFrame({"line": lines, "used": usage @ x_val, "capacity": capacity})

display(production, lines_used)

,chair,quantity,profit
0,A,5.25,21.0
1,B,5.00,30.0


,line,used,capacity
0,Line 1,10.5,14
1,Line 2,15.0,15
2,Line 3,36.0,36


In [12]:
rng = np.random.default_rng(42)
chairs10 = [f"c{j+1}" for j in range(10)]
lines5 = [f"p{i+1}" for i in range(5)]

profit10 = rng.integers(3, 12, size=10)
usage10 = rng.integers(0, 6, size=(5, 10))
capacity5 = rng.integers(40, 100, size=5)

model, x, result = solve_chairs(profit10, capacity5, usage10, chairs10, lines5)
print(result.status_str, result.objective)

Optimal 237.86824324324326


In [14]:
x_val = np.array([x[c].value() for c in chairs10])

production = pd.DataFrame({"chair": chairs10, "quantity": x_val, "profit": profit10 * x_val})
lines_used = pd.DataFrame({"line": lines5, "used": usage10 @ x_val, "capacity": capacity5})

display(production, lines_used)

,chair,quantity,profit
0,c1,0.000000,0.000000
1,c2,0.000000,0.000000
2,c3,8.368243,66.945946
3,c4,0.000000,0.000000
4,c5,7.878378,47.270270
5,c6,7.503378,75.033784
6,c7,0.000000,0.000000
7,c8,5.402027,48.618243
8,c9,0.000000,0.000000
9,c10,0.000000,0.000000


,line,used,capacity
0,p1,95.000000,95
1,p2,84.000000,84
2,p3,61.000000,61
3,p4,98.000000,98
4,p5,32.493243,64
